# 2024 Tip of the Tongue Track

The project works with the TREC 2024 Tip of the Tongue dataset.

https://pages.nist.gov/trec-browser/trec33/tot/data/

The dataset for the 2024 Tip-of-the-Tongue task contains queries, a shared corpus, and qrels. The queries are in JSONL format, qrels are in TXT format, and the documents are obtained from the official ToT corpus via ir_datasets. We need to perform a retrieval task where we have to retrieve the correct target document from the shared corpus for the query.


In [1]:
# data loading: TREC 2024 Tip-of-the-Tongue
!pip install -U ir_datasets

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.8 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=57ed31f20e5b7350f847b820635a5780c7874a193ea3ce593cf9988e2a024d5a
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c
  Created wheel for cbor: filename=cbor-1.0.0-cp312-cp312-linux_x86_64.whl size=55021 sha256=d616e0fd16c57fda93e7c858c84f21760756b9f67905ee467ddcfa897afad8c8
  Stored in directory: /root/.cache/pip/wheels/44/3e/21/a739cbcc331a1ab45c326d6edbdac6118de4402f6076e30ff1
Successfully built warc3-wet-clueweb09 cbor


In [2]:
# import libraries
from google.colab import drive
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ir_datasets
import os
from itertools import islice
import glob
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import re
import urllib.request
import zipfile

In [3]:
# loading corpus
corpus_2024=ir_datasets.load("trec-tot/2024")
print(corpus_2024)

Dataset(id='trec-tot/2024', provides=['docs'])


In [4]:
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/unzipped", exist_ok=True)

files = {
    "train": "https://zenodo.org/records/13370657/files/train-2024.zip?download=1",
    "dev1":  "https://zenodo.org/records/13370657/files/dev1-2024.zip?download=1",
    "dev2":  "https://zenodo.org/records/13370657/files/dev2-2024.zip?download=1",
}

for name, url in files.items():
    zip_path = f"data/raw/{name}.zip"

    if not os.path.exists(zip_path):
        print(f"Downloading {name}...")
        urllib.request.urlretrieve(url, zip_path)
    else:
        print(f"{name} already exists.")

    # unzip
    extract_path = f"data/unzipped/{name}"
    os.makedirs(extract_path, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("Download + unzip completed")

# dev1 validation set
# dev2 test set

Download + unzip completed


In [8]:
# this block searches for query and qrel files inside the train and dev

# locate training queries
train_query_list = glob.glob("data/unzipped/train/**/*.jsonl", recursive=True)
train_qrel_list  = glob.glob("data/unzipped/train/**/qrel.txt", recursive=True)

# locate validation queries
val_query_list = glob.glob("data/unzipped/dev1/**/*.jsonl", recursive=True)
val_qrel_list  = glob.glob("data/unzipped/dev1/**/qrel.txt", recursive=True)

# locate test queries
test_query_list = glob.glob("data/unzipped/dev2/**/*.jsonl", recursive=True)
test_qrel_list  = glob.glob("data/unzipped/dev2/**/qrel.txt", recursive=True)

print("train query candidates:", train_query_list)
print("train qrel candidates :", train_qrel_list)
print("val query candidates  :", val_query_list)
print("val qrel candidates   :", val_qrel_list)
print("test query candidates :", test_query_list)
print("test qrel candidates  :", test_qrel_list)

train_query_path = train_query_list[0]
train_qrel_path  = train_qrel_list[0]

val_query_path = val_query_list[0]
val_qrel_path  = val_qrel_list[0]

test_query_path = test_query_list[0]
test_qrel_path  = test_qrel_list[0]

print("Train query path:", train_query_path)
print("Train qrel path :", train_qrel_path)
print("Validation query path:", val_query_path)
print("Validation qrel path :", val_qrel_path)
print("Test query path :", test_query_path)
print("Test qrel path  :", test_qrel_path)

train query candidates: ['data/unzipped/train/train-2024/queries.jsonl']
train qrel candidates : ['data/unzipped/train/train-2024/qrel.txt']
val query candidates  : ['data/unzipped/dev1/dev1-2024/queries.jsonl']
val qrel candidates   : ['data/unzipped/dev1/dev1-2024/qrel.txt']
test query candidates : ['data/unzipped/dev2/dev2-2024/queries.jsonl']
test qrel candidates  : ['data/unzipped/dev2/dev2-2024/qrel.txt']
Train query path: data/unzipped/train/train-2024/queries.jsonl
Train qrel path : data/unzipped/train/train-2024/qrel.txt
Validation query path: data/unzipped/dev1/dev1-2024/queries.jsonl
Validation qrel path : data/unzipped/dev1/dev1-2024/qrel.txt
Test query path : data/unzipped/dev2/dev2-2024/queries.jsonl
Test qrel path  : data/unzipped/dev2/dev2-2024/qrel.txt


In [9]:
# loading query datasets for train and dev splits
# this query represents a user information retrieval
queries_train_2024 = pd.read_json(train_query_path, lines=True)
queries_val_2024  = pd.read_json(val_query_path, lines=True)
queries_test_2024  = pd.read_json(test_query_path, lines=True)

# both cases have 150 queries and 2 columns (query_id, query)
print("Train queries shape:", queries_train_2024.shape)
print("Validation queries shape :", queries_val_2024.shape)
print("Test queries shape :", queries_test_2024.shape)

# to understand context, we display first 5 columns
print(queries_train_2024.head())
print(queries_val_2024.head())
print(queries_test_2024.head())

Train queries shape: (150, 2)
Validation queries shape : (150, 2)
Test queries shape : (150, 2)
   query_id                                              query
0       763  Super Rare Surreal Dystopian Masterpiece .\n V...
1       802  Male little person falls in love with red-head...
2       950  Movie about two girls who run away and murder ...
3       220  Possibly an occult film .\n I remember I was y...
4       792  kid who robs houses .\n Ok, so I seen this mov...
   query_id                                              query
0       152  Foriegn Film about 3 Strangers in an Apartment...
1       531  I need help .\n Alright so I saw this movie so...
2       473  horror movie with a old lady , possibly a ghos...
3       659  Funny dads- dvd cover of dads going down water...
4      1095  Fantasy movie involving a giant computer? .\n ...
   query_id                                              query
0       519  redbox mystery driving me insane .\n Ok, so my...
1      1006  Girl in a

These queries reflect real-world user behavior.These queries often expressed as full sentences and detailed explanation. Lexical models such as BM25 are still important for initial retrieval, while semantic models like SBERT and MonoT5 are more effective for capturing contextual meaning and reranking.

In [10]:
# Qrels provide ground-truth labels that map queries to
# relevant documents. They are used for evaluation and training
# of retrieval models.

qrels_train_2024 = pd.read_csv(
    train_qrel_path,
    sep=r"\s+",
    names=["query_id", "iteration", "doc_id", "relevance"],
    engine="python"
)

qrels_val_2024 = pd.read_csv(
    val_qrel_path,
    sep=r"\s+",
    names=["query_id", "iteration", "doc_id", "relevance"],
    engine="python"
)

qrels_test_2024 = pd.read_csv(
    test_qrel_path,
    sep=r"\s+",
    names=["query_id", "iteration", "doc_id", "relevance"],
    engine="python"
)

print("Train qrels shape:", qrels_train_2024.shape)
print("Validation qrels shape :", qrels_val_2024.shape)
print("Test qrels shape :", qrels_test_2024.shape)

print(qrels_train_2024.head())
print(qrels_val_2024.head())
print(qrels_test_2024.head())

Train qrels shape: (150, 4)
Validation qrels shape : (150, 4)
Test qrels shape : (150, 4)
   query_id  iteration    doc_id  relevance
0       763          0  16742289          1
1       802          0  30523669          1
2       950          0   1705452          1
3       220          0   4891218          1
4       792          0   4815950          1
   query_id  iteration    doc_id  relevance
0       152          0   1940119          1
1       531          0  30199617          1
2       473          0  15770244          1
3       659          0  35867054          1
4      1095          0  37314908          1
   query_id  iteration    doc_id  relevance
0       519          0  28286160          1
1      1006          0  17905510          1
2       477          0  24073089          1
3       528          0  12095072          1
4       662          0  32467228          1


In [11]:
queries_2024_train = queries_train_2024.copy()
queries_2024_val = queries_val_2024.copy()
queries_2024_test = queries_test_2024.copy()

qrels_2024_train = qrels_train_2024.copy()
qrels_2024_val = qrels_val_2024.copy()
qrels_2024_test = qrels_test_2024.copy()

In [12]:
# total documents count is 3185450.
count = 0
for _ in corpus_2024.docs_iter():
    count += 1

print("Total documents:", count)

corpus_head = list(islice(corpus_2024.docs_iter(), 5))
corpus_df = pd.DataFrame(corpus_head)

print("Corpus head shape:", corpus_df.shape)
print(corpus_df)

[INFO] [starting] opening zip file
[INFO] If you have a local copy of https://zenodo.org/records/13370657/files/corpus.jsonl.zip?download=1, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/4ea86770817e46a06fea5c94f596409c
[INFO] [starting] https://zenodo.org/records/13370657/files/corpus.jsonl.zip?download=1
[INFO] [finished] https://zenodo.org/records/13370657/files/corpus.jsonl.zip?download=1: [02:32] [3.13GB] [20.5MB/s]
[INFO] [finished] opening zip file [02:33]


Total documents: 3185450
Corpus head shape: (5, 5)
  doc_id                    title wikidata_id  \
0    846           Museum of Work    Q6941060   
1    340             Alain Connes     Q313590   
2    868               Alp Arslan     Q200193   
3    869  American Film Institute     Q207460   
4    628            Aldous Huxley      Q81447   

                                                text  \
0  The Museum of Work ("Arbetets museum") is a mu...   
1  Alain Connes (; born 1 April 1947) is a French...   
2  Alp Arslan was the second Sultan of the Seljuk...   
3  The American Film Institute (AFI) is an Americ...   
4  Aldous Leonard Huxley (26 July 1894 – 22 Novem...   

                                            sections  
0  [{'start': 0, 'end': 798, 'section': 'Abstract...  
1  [{'start': 0, 'end': 300, 'section': 'Abstract...  
2  [{'start': 0, 'end': 475, 'section': 'Abstract...  
3  [{'start': 0, 'end': 238, 'section': 'Abstract...  
4  [{'start': 0, 'end': 1378, 'section': '

The data set has two major parts: the training set and the development set. The training set is used for retrieval model building and tuning, and the development set is used for evaluation. Both the training set and the development set have several entries of text with corresponding relevance judgments. The text entries are descriptions of items, like movies, written by users in an incomplete and vague manner. The goal of the task is to retrieve the correct item from the set of candidates.

# EDA and Quality Checks
